In [ ]:
import numpy as np
import pandas as pd
def calcular_porcentaje_wrapping_l1(num_muestras=10000000):
    # Fijamos L = 1
    L = 1.0
    
    # Probamos diferentes radios de kernel en proporción a L
    radios_prop = [0.5, 0.7, 1.0, 1.25, 1.5, 2.0, 4.0]
    
    resultados = []
    
    # Inicializar generador aleatorio para reproducibilidad
    rng = np.random.default_rng(42)
    
    for r_eff in radios_prop:
        # 1. Distancia máxima a la que el kernel puede tocar la hoja (vértice + r)
        d_max = (L * np.sqrt(2)) + r_eff
        
        # Generar puntos aleatorios en el plano XY alrededor de la hoja
        # Para ser eficientes y uniformes, generamos en un cuadrado amplio y filtramos
        x = rng.uniform(-d_max, d_max, num_muestras)
        y = rng.uniform(-d_max, d_max, num_muestras)
        
        dxy_sq = x**2 + y**2
        dxy = np.sqrt(dxy_sq)
        
        # Filtro A: La query debe estar dentro del rango de influencia máxima
        # Filtro B: Para que tenga sentido evaluar la hoja, el kernel debe tocar el cuadrado de la hoja.
        # Distancia aproximada al cuadrado: aproximamos por intersección círculo-cuadrado
        
        # Distancia mínima desde la query (x,y) al cuadrado de la hoja [-0.5, 0.5]
        dx_cuadrado = np.maximum(0, np.abs(x) - L)
        dy_cuadrado = np.maximum(0, np.abs(y) - L)
        dist_al_cuadrado = np.sqrt(dx_cuadrado**2 + dy_cuadrado**2)
        
        # Máscara de queries que realmente intersecan la hoja
        interseca_hoja = dist_al_cuadrado <= r_eff
        
        # Nos quedamos solo con las queries válidas que tocan la hoja
        x_valid = x[interseca_hoja]
        y_valid = y[interseca_hoja]
        dxy_valid = dxy[interseca_hoja]
        
        if len(dxy_valid) == 0:
            continue
            
        # 2. Calcular variables geométricas angulares
        phi_Q = np.arctan2(y_valid, x_valid)
        phi_Q = np.where(phi_Q < 0, phi_Q + 2 * np.pi, phi_Q) # Normalizar 0 a 2pi
        
        # Calcular deltaPhi con el clamp de tu código
        ratio = r_eff / dxy_valid
        ratio_clamped = np.clip(ratio, 0.0, 1.0)
        delta_phi = np.arcsin(ratio_clamped)
        
        # 3. Evaluar la condición de corte (Wrapping)
        k_min_raw = phi_Q - delta_phi
        k_max_raw = phi_Q + delta_phi
        
        obligan_full = (k_min_raw < 0.0) | (k_max_raw >= 2 * np.pi)
        
        # Calcular porcentaje
        total_intersecciones = len(dxy_valid)
        total_wrapping = np.sum(obligan_full)
        porcentaje = (total_wrapping / total_intersecciones) * 100
        
        resultados.append({
            "Radio Kernel (R/L)": r_eff,
            "Distancia Máx (D_max/L)": round(d_max, 3),
            "Queries Analizadas": total_intersecciones,
            "Hojas con Wrapping": total_wrapping,
            "Porcentaje de FULL (%)": round(porcentaje, 2)
        })
        
    return pd.DataFrame(resultados)

# Ejecutar el cálculo
df_tabla = calcular_porcentaje_wrapping_l1()
print(df_tabla.to_string(index=False))

 Radio Kernel ($r/L$)  Distancia Máx ($d_{max}/L$)  Queries Analizadas  Hojas con Wrapping  Porcentaje de FULL (%)
                 0.50                        1.914             5995385             1022314                   17.05
                 0.70                        2.114             6230098             1332525                   21.39
                 1.00                        2.414             6492525             1715006                   26.42
                 1.25                        2.664             6658287             1979401                   29.73
                 1.50                        2.914             6790485             2201139                   32.42
                 2.00                        3.414             6984847             2538304                   36.34
                 4.00                        5.414             7358767             3200470                   43.49


In [ ]:
df_tabla

,Radio Kernel ($r/L$),Distancia Máx ($d_{max}/L$),Queries Analizadas,Hojas con Wrapping,Porcentaje de FULL (%)
0,0.50,1.914,5995385,1022314,17.05
1,0.70,2.114,6230098,1332525,21.39
2,1.00,2.414,6492525,1715006,26.42
3,1.25,2.664,6658287,1979401,29.73
4,1.50,2.914,6790485,2201139,32.42
5,2.00,3.414,6984847,2538304,36.34
6,4.00,5.414,7358767,3200470,43.49
